# Explore downloaded facilities and their attributes as well as aggregated facilities

In [41]:
import pandas as pd

In [42]:
from src.co2sat.utils import project_root

In [43]:
sample = pd.read_csv(
    project_root() / "data" / "raw" / "epa_facilities" / "facility-2021.csv",
    nrows=5,
)

In [44]:
print("Columns:", sample.columns.T)

Columns: Index(['State', 'Facility Name', 'Facility ID', 'Unit ID', 'Associated Stacks',
       'Year', 'Program Code', 'Primary Rep Info', 'EPA Region', 'NERC Region',
       'County', 'County Code', 'FIPS Code', 'Source Category', 'Latitude',
       'Longitude', 'Owner/Operator', 'SO2 Phase', 'NOx Phase', 'Unit Type',
       'Primary Fuel Type', 'Secondary Fuel Type', 'SO2 Controls',
       'NOx Controls', 'PM Controls', 'Hg Controls',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)'],
      dtype='str')


In [45]:
sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 30 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   State                                             5 non-null      str    
 1   Facility Name                                     5 non-null      str    
 2   Facility ID                                       5 non-null      int64  
 3   Unit ID                                           5 non-null      str    
 4   Associated Stacks                                 2 non-null      str    
 5   Year                                              5 non-null      int64  
 6   Program Code                                      5 non-null      str    
 7   Primary Rep Info                                  5 non-null      int64  
 8   EPA Region                                        5 non-null      int64  
 9   NERC Region                         

In [46]:
print(sample["Associated Generators & Nameplate Capacity (MWe)"].head(20))

0                     1 (153.1)
1                     2 (153.1)
2                     4 (403.8)
3                     5 (788.8)
4    A1ST (191.8), A1CT (170.1)
Name: Associated Generators & Nameplate Capacity (MWe), dtype: str


In [47]:
print(
    sample["Associated Generators & Nameplate Capacity (MWe)"]
    .dropna()
    .sample(5)
    .tolist()
)

['A1ST (191.8), A1CT (170.1)', '1 (153.1)', '2 (153.1)', '4 (403.8)', '5 (788.8)']


Note the capacities have to be parsed from the format shown above. This is why we imported the function `parse_capacities` in `src/co2sat/data/epa.py`

In [48]:
raw_dir = project_root() / "data" / "raw" / "epa_facilities"

In [49]:
sample = pd.read_csv(
    raw_dir / "facility-2021.csv",
    usecols=["Associated Generators & Nameplate Capacity (MWe)"],
)
cap_col = sample["Associated Generators & Nameplate Capacity (MWe)"]

Let's test to parse capacities in the `2021` file

In [50]:
print(f"Total rows:       {len(cap_col):,}")

Total rows:       4,266


In [51]:
print(f"Null values:      {cap_col.isnull().sum():,}")

Null values:      173


In [52]:
print(f"Non-null values:  {cap_col.notna().sum():,}")

Non-null values:  4,093


In [53]:
# Strings without any parenthesized number
non_matching = cap_col.dropna()[
    ~cap_col.dropna().str.contains(r"\([\d.]+\)", regex=True)
]

In [54]:
print(f"\nNon-null strings with no parenthesized number: {len(non_matching)}")


Non-null strings with no parenthesized number: 0


We confirm that all capacity units are written beteween parentheses

Now let's test the parsing capacity function in a sample

In [55]:
from src.co2sat.data.epa import parse_capacities

In [56]:
test = cap_col.dropna().sample(20, random_state=42)
for s in test:
    print(f"{s[:60]:60s} -> {parse_capacities(s):.1f} MW")

GT93 (278.6), ST96 (315)                                     -> 593.6 MW
1 (56.7)                                                     -> 56.7 MW
2 (170)                                                      -> 170.0 MW
6 (27.2), 5 (30.5), 7 (47.3)                                 -> 105.0 MW
GEN3 (27.8), GEN2 (48.8)                                     -> 76.6 MW
CTG1 (47.3)                                                  -> 47.3 MW
2 (224)                                                      -> 224.0 MW
4 (375), CT4B (232.5)                                        -> 607.5 MW
3 (750)                                                      -> 750.0 MW
2 (165)                                                      -> 165.0 MW
2 (60)                                                       -> 60.0 MW
4 (825)                                                      -> 825.0 MW
CT1 (63)                                                     -> 63.0 MW
103G (176)                                              

We confirm that the capacity parser function handles all encountered formats, or a list of edge cases to handle

Now we notice that there is an `Operating Status` column in the downloaded datasets which is good to explore

In [57]:
sample = pd.read_csv(raw_dir / "facility-2021.csv", usecols=["Operating Status"])
print(sample["Operating Status"].value_counts(dropna=False))

Operating Status
Operating                         4148
Retired                             27
Operating (Retired 04/01/2021)      10
Operating (Retired 06/01/2021)       9
Operating (Retired 05/31/2021)       5
Long-term Cold Storage               4
Operating (Retired 10/01/2021)       4
Operating (Retired 07/30/2021)       4
Operating (Retired 03/31/2021)       3
Operating (Started 08/04/2021)       3
Operating (Started 07/12/2021)       3
Operating (Started 09/17/2021)       2
Operating (Retired 12/01/2021)       2
Operating (Retired 04/06/2021)       2
Operating (Started 05/27/2021)       2
Operating (Started 09/01/2021)       2
Operating (Started 07/05/2021)       2
Operating (Started 07/08/2021)       2
Operating (Started 10/23/2021)       1
Operating (Started 10/01/2021)       1
Operating (Started 11/07/2021)       1
Operating (Retired 11/30/2021)       1
Operating (Started 09/28/2021)       1
Operating (Started 09/21/2021)       1
Operating (Retired 04/04/2021)       1
Operatin

Clean values (no parenthetical):

Operating — 4,148 units, currently running. Keep.

Retired — 27 units, no longer operating. Drop.

Long-term Cold Storage — 4 units, mothballed. Drop.

Future — 1 unit, not built yet. Drop.

To preserve this logic whatever the year is we created a function in `src/co2sat/data/epa.py` to proceed later with filtering. We call this function `is_active`

Now it is time to define multiple functions in `epa.py` in order to load, filter and aggregate the download facilities files with the files of daily emissions

In [58]:
facilities = pd.read_parquet(
    project_root() / "data" / "processed" / "epa_facilities.parquet"
)

In [59]:
print(f"Total facilities: {len(facilities):,}")
print()
print("=== Coordinate completeness ===")
print(f"With lat/lon:    {facilities[['latitude', 'longitude']].dropna().shape[0]:,}")
print(f"Missing lat/lon: {facilities['latitude'].isnull().sum():,}")
print()
print("=== Capacity ===")
print(f"With capacity > 0:       {(facilities['capacity_mw'] > 0).sum():,}")
print(f"With capacity == 0/null: {(facilities['capacity_mw'] <= 0).sum():,}")
print(
    f"Capacity range:          {facilities['capacity_mw'].min():.1f} - {facilities['capacity_mw'].max():.1f} MW"
)
print(f"Median capacity:         {facilities['capacity_mw'].median():.1f} MW")
print()
print("=== Fuel ratios ===")
ratio_cols = ["coal_ratio", "gas_ratio", "oil_ratio", "other_ratio"]
sums = facilities[ratio_cols].sum(axis=1).round(2)
print(f"Facilities with ratios summing to 1.0:   {(sums == 1.0).sum():,}")
print(f"Facilities with ratios summing to 0.0:   {(sums == 0.0).sum():,}")
print(f"Facilities with other sum:               {(~sums.isin([0.0, 1.0])).sum():,}")
print()
print("=== Primary fuel distribution ===")
print(facilities["primary_fuel"].value_counts(dropna=False).head(15))
print()
print("=== Operating status distribution ===")
print(facilities["operating_status"].value_counts(dropna=False).head(10))
print()
print("=== State distribution (top 10) ===")
print(facilities["state"].value_counts().head(10))
print()
print("=== Schema ===")
print(facilities.dtypes)
print()
print("=== Sample rows ===")
print(facilities.head(5))

Total facilities: 1,400

=== Coordinate completeness ===
With lat/lon:    1,400
Missing lat/lon: 0

=== Capacity ===
With capacity > 0:       1,349
With capacity == 0/null: 51
Capacity range:          0.0 - 8669.4 MW
Median capacity:         388.1 MW

=== Fuel ratios ===
Facilities with ratios summing to 1.0:   1,400
Facilities with ratios summing to 0.0:   0
Facilities with other sum:               0

=== Primary fuel distribution ===
primary_fuel
Pipeline Natural Gas          980
Coal                          208
Diesel Oil                     76
Natural Gas                    52
Wood                           21
Process Gas                    17
Residual Oil                   11
Other Gas                       9
Other Oil                       8
Coal Refuse                     7
<NA>                            6
Petroleum Coke                  4
Coal, Pipeline Natural Gas      1
Name: count, dtype: int64[pyarrow]

=== Operating status distribution ===
operating_status
Operating     

Now that wa have joined facilities to daily emission files we can check their numbers in comparison with the paper

In [60]:
PERIODS = [
    ("2021-04-01", "2021-05-20"),
    ("2021-09-01", "2021-10-01"),
    ("2022-04-01", "2022-04-30"),
    ("2022-09-01", "2022-09-29"),
]
PAPER_PLANTS = [583, 590, 513, 592]

In [61]:
processed_dir = project_root() / "data" / "processed"

In [62]:
joined = pd.read_parquet(processed_dir / "epa_daily_with_attributes.parquet")

In [63]:
joined["date"] = pd.to_datetime(joined["date"])

In [64]:
print(f"{'Period':30s} {'Plants':>8s} {'(paper)':>8s} {'Δ':>6s} {'Rows':>7s}")
print("-" * 70)
for (start, end), pp in zip(PERIODS, PAPER_PLANTS):
    sub = joined[(joined["date"] >= start) & (joined["date"] < end)]
    print(
        f"  {start} to {end[:7]:8s}  "
        f"{sub['facility_id'].nunique():8d} {pp:8d} "
        f"{sub['facility_id'].nunique() - pp:+5d} {len(sub):7d}"
    )

Period                           Plants  (paper)      Δ    Rows
----------------------------------------------------------------------
  2021-04-01 to 2021-05       1038      583  +455   29525
  2021-09-01 to 2021-10       1066      590  +476   21696
  2022-04-01 to 2022-04        986      513  +473   17607
  2022-09-01 to 2022-09       1046      592  +454   21433


In [65]:
joined.info()

<class 'pandas.DataFrame'>
RangeIndex: 507574 entries, 0 to 507573
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   facility_id        507574 non-null  int64         
 1   state              507574 non-null  str           
 2   date               507574 non-null  datetime64[us]
 3   co2_metric_tons    507574 non-null  float64       
 4   gross_load_mwh     507574 non-null  float64       
 5   primary_fuel_type  507574 non-null  str           
 6   facility_name      507574 non-null  string        
 7   latitude           507574 non-null  Float64       
 8   longitude          507574 non-null  Float64       
 9   capacity_mw        507574 non-null  float64       
 10  primary_fuel       507574 non-null  string        
 11  operating_status   507574 non-null  string        
 12  n_units            507574 non-null  float64       
 13  year               507574 non-null  Int64         
 14 

In [66]:
joined["operating_status"].value_counts(dropna=False)

operating_status
Operating                         503168
Operating (Retired 06/30/2022)       614
Operating (Retired 11/28/2022)       535
Operating (Retired 05/31/2022)       489
Operating (Retired 09/30/2022)       484
Operating (Retired 07/15/2022)       445
Operating (Retired 04/01/2022)       417
Operating (Started 03/10/2022)       216
Operating (Retired 08/31/2022)       181
Operating (Retired 06/14/2022)       175
Operating (Retired 06/01/2021)       161
Operating (Retired 08/19/2022)       153
Operating (Retired 06/01/2022)       148
Operating (Retired 04/11/2021)       101
Operating (Retired 04/01/2021)        83
Operating (Retired 09/01/2022)        54
Operating (Started 05/25/2022)        43
Operating (Retired 06/02/2022)        33
Operating (Retired 11/01/2022)        29
Operating (Started 09/26/2022)        20
Operating (Started 08/26/2022)        19
Operating (Retired 03/31/2021)         6
Name: count, dtype: int64[pyarrow]

In [67]:
# Example: a facility with status "Operating (Retired 06/30/2022)"
# Find one such facility
retired_2022_06 = joined[
    joined["operating_status"] == "Operating (Retired 06/30/2022)"
]["facility_id"].unique()

# Pick the first one
fid = retired_2022_06[0]
print(f"Looking at facility {fid}\n")

# Look at all its daily data, sorted by date
plant_data = joined[joined["facility_id"] == fid][
    ["date", "co2_metric_tons", "gross_load_mwh"]
].sort_values("date")

print(plant_data.to_string())

Looking at facility 2451

            date  co2_metric_tons  gross_load_mwh
93375 2021-01-01     40735.134943        37774.00
93376 2021-01-02     40261.765946        37344.00
93377 2021-01-03     40726.063096        38316.00
93378 2021-01-04     40793.557641        38060.00
93379 2021-01-05     39749.388005        37080.00
93380 2021-01-06     39530.105124        36847.72
93381 2021-01-07     25206.516276        22550.96
93382 2021-01-08     37308.698180        34804.00
93383 2021-01-09     41197.436287        39100.00
93384 2021-01-10     42358.814191        40128.00
93385 2021-01-11     41343.311593        39012.00
93386 2021-01-12     41537.449127        39498.00
93387 2021-01-13     37501.565656        36144.00
93388 2021-01-14     39090.227573        38768.00
93389 2021-01-15     38965.580389        38726.00
93390 2021-01-16     39082.425784        38384.00
93391 2021-01-17     38760.193764        37972.00
93392 2021-01-18     40307.488057        39398.00
93393 2021-01-19     403

In [68]:
import re
from datetime import datetime

RETIRED_RE = re.compile(r"Operating \(Retired (\d{2}/\d{2}/\d{4})\)")
STARTED_RE = re.compile(r"Operating \(Started (\d{2}/\d{2}/\d{4})\)")

In [69]:
def operating_window(status: str) -> tuple[datetime | None, datetime | None]:
    """Return (start_date, end_date) for an EPA operating status string.

    None means open-ended in that direction. So:
    - "Operating" -> (None, None) — always operating
    - "Operating (Started 03/10/2022)" -> (2022-03-10, None) — started, no end
    - "Operating (Retired 06/30/2022)" -> (None, 2022-06-30) — operating until retirement
    """
    if not status:
        return (None, None)

    m_retired = RETIRED_RE.match(status)
    if m_retired:
        end = datetime.strptime(m_retired.group(1), "%m/%d/%Y")
        return (None, end)

    m_started = STARTED_RE.match(status)
    if m_started:
        start = datetime.strptime(m_started.group(1), "%m/%d/%Y")
        return (start, None)

    return (None, None)

In [70]:
# Apply to your joined data
def in_operating_window(row) -> bool:
    start, end = operating_window(row["operating_status"])
    date = row["date"]
    if start is not None and date < start:
        return False
    if end is not None and date > end:
        return False
    return True

In [71]:
joined["in_window"] = joined.apply(in_operating_window, axis=1)
filtered = joined[joined["in_window"]].copy()

In [72]:
print(f"{'Period':30s} {'Plants':>8s} {'(paper)':>8s} {'Δ':>6s} {'Rows':>7s}")
print("-" * 70)
for (start, end), pp in zip(PERIODS, PAPER_PLANTS):
    sub = filtered[(filtered["date"] >= start) & (filtered["date"] < end)]
    print(
        f"  {start} to {end[:7]:8s}  "
        f"{sub['facility_id'].nunique():8d} {pp:8d} "
        f"{sub['facility_id'].nunique() - pp:+5d} {len(sub):7d}"
    )

Period                           Plants  (paper)      Δ    Rows
----------------------------------------------------------------------
  2021-04-01 to 2021-05       1038      583  +455   29525
  2021-09-01 to 2021-10       1066      590  +476   21696
  2022-04-01 to 2022-04        986      513  +473   17607
  2022-09-01 to 2022-09       1045      592  +453   21405
